In [1]:
from torchvision.datasets import FashionMNIST
import torchvision
from torchvision.transforms import v2
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DECICE USED:", device)

train_tf = v2.Compose([
    v2.RandomCrop(28, padding=2),
    v2.RandomHorizontalFlip(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.2860,), (0.3530,)),
    v2.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

test_tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.2860,), (0.3530,)),
])

train_data = FashionMNIST(root="data", train=True, download=True,
                        transform=train_tf,
                        target_transform=None)
test_data = FashionMNIST(root="data", train=False, download=True,
                        transform=test_tf,
                        target_transform=None)

DECICE USED: cuda


100%|██████████| 26.4M/26.4M [00:01<00:00, 13.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 3.94MB/s]


In [ ]:
import matplotlib.pyplot as plt

class_names = train_data.classes
fig, ax = plt.subplots(4,4,figsize=(8,8))
k = 0
for i in range(4):
    for j in range(4):
        image, label = train_data[k]
        ax[i, j].imshow(image.squeeze(), cmap='grey')
        ax[i,j].axis(False)
        ax[i,j].set_title(class_names[int(label)])
        k += 1

In [ ]:
from torch.utils.data import DataLoader
import torch
torch.manual_seed(152)
BATCH_SIZE = 32
train_dataloader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(dataset=test_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
from sklearn.metrics import accuracy_score

def train_step(model: torch.nn.Module, data_loader, loss_fn, optimizer,):
     loss_total = 0
     acc_total = 0
     model.train()
     for X, y in data_loader:
        X = X.to(device)
        y = y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        loss_total += loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        acc_total += accuracy_score(y.cpu(), y_pred.cpu().argmax(dim=1))
     loss_total /= len(data_loader)
     acc_total /= len(data_loader)
     return {"Loss": loss_total.item(), "Acc": acc_total}

def test_step(model: torch.nn.Module, data_loader, loss_fn):
    loss_total = 0
    acc_total = 0
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            X = X.to(device)
            y = y.to(device)
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            loss_total += loss
            acc_total += accuracy_score(y.cpu(), y_pred.cpu().argmax(dim=1))
        loss_total /= len(data_loader)
        acc_total /= len(data_loader)
        return {"Loss": loss_total.item(), "Acc": acc_total}

In [ ]:
from torch import nn
from torchvision import transforms

class FashionClassifier(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1,
                        padding=1),
            nn.BatchNorm2d(num_features=hidden_units),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1,
                        padding=1),
            nn.BatchNorm2d(num_features=hidden_units),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2))

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units*2,
                        kernel_size=3,
                        stride=1,
                        padding=1),
            nn.BatchNorm2d(num_features=hidden_units*2),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units*2,
                        out_channels=hidden_units*2,
                        kernel_size=3,
                        stride=1,
                        padding=1),
            nn.BatchNorm2d(num_features=hidden_units*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2))

        self.classifier_layer = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(in_features=hidden_units*2*7*7,
                      out_features=output_shape))

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier_layer(x)
        return x


In [ ]:
model = FashionClassifier(input_shape=1, hidden_units=10, output_shape=len(class_names)).to(device)

In [ ]:
from tqdm import tqdm

epochs = 20
optimizer = torch.optim.AdamW(params=model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
train_loss_list = list()
test_loss_list = list()

for epoch in tqdm(range(epochs)):
    res_train = train_step(model=model,
                           data_loader=train_dataloader,
                           loss_fn=loss_fn,
                           optimizer=optimizer)
    train_loss_list.append(res_train['Loss'])
    res_test = test_step(model=model,
                           data_loader=test_dataloader,
                           loss_fn=loss_fn)
    test_loss_list.append(res_test['Loss'])
    print(f"EPOCH: {epoch} \n TRAIN Loss: {res_train['Loss']} Acc: {res_train['Acc']*100:.2f}% \n TEST Loss: {res_test['Loss']} Acc: {res_test['Acc']*100:.2f}%")

In [ ]:
fig, ax = plt.subplots()
ax.plot(train_loss_list)
ax.plot(test_loss_list)